In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    r"C:\Users\HELIOS-300\Desktop\WAVES\ACT24 Full Code\Cameron_ACT24_Clean_NoDrop.csv",
    low_memory=False
).copy()

print(df.shape)
print(df.columns.tolist())
print(df["activity_type"].value_counts(dropna=False).head(20))


(490080, 15)
['id', 'obs', 'date', 'date_time', 'rel_time', 'activity_type', 'broad_domain', 'broad.behavior_do', 'posture_wbm', 'posture_broad', 'broad.posture_do', 'sed.posture_do', 'intensity_do', 'Quality', 'Step']
activity_type
work_screen      113149
ex_sport          82514
work_general      67168
ha_eat            28638
les_screen        26188
trav_drive        23158
care_children     22249
edu_class         18606
ha_other          17764
trav_walk         15996
les_social        14562
ha_food           14312
non_codable       12418
com_purchase       6984
trav_pass          6655
sleep              5953
ha_housework       3109
com_volunteer      2656
pc_groom           2309
ha_pets            2245
Name: count, dtype: int64


In [ ]:
# Step 1: Convert coded activity_type -> full label (backwards mapping)
coded_to_label = {
    "sleep":         "SL- Sleep",
    "pc_groom":      "PC- Groom, Health-Related",
    "pc_other":      "PC- Other Personal Care",
    "ha_housework":  "HA- Housework",
    "ha_food":       "HA- Food Prep and Cleanup",
    "ha_interior":   "HA- Interior Maintenance, Repair, & Decoration",
    "ha_exterior":   "HA- Exterior Maintenance, Repair, & Decoration",
    "ha_lawn":       "HA- Lawn, Garden and Houseplants",
    "ha_pets":       "HA- Animals and Pets",
    "ha_other":      "HA- Household Management/Other Household Activities",
    "care_children": "CA- Caring for and Helping Children",
    "care_adults":   "CA- Caring for and Helping Adults",
    "work_general":  "WRK- General",
    "work_screen":   "WRK- Desk/Screen Based",
    "edu_class":     "EDU- Taking Class, Research, Homework",
    "edu_other":     "EDU- Extracurricular",
    "com_church":    "ORG- Church, Spiritual",
    "com_volunteer": "ORG- Volunteer Work",
    "com_purchase":  "PUR- Purchasing Goods and Services",
    "ha_eat":        "EAT- Eating and Drinking, Waiting",
    "les_social":    "LES- Socializing, Communicating, Non-Screen Based",
    "les_screen":    "LES- Screen-Based (TV, Video Game, Computer, Phone)",
    "ex_sport":      "EX- Participating in Sport, Exercise or Recreation",
    "les_attend":    "EX- Attending Sport, Exercise Recreation Event, or Performance",
    "trav_pass":     "TRAV- Passenger (Car/Truck/Motorcycle)",
    "trav_drive":    "TRAV- Driver (Car/Truck/Motorcycle)",
    "trav_bike":     "TRAV- Biking",
    "trav_walk":     "TRAV- Walking",
    "trav_other":    "TRAV- General",
    "non_codable":   "OTHER- Non-Codable",
}

df["activity_type"] = df["activity_type"].map(coded_to_label)

# Step 2: Derive broad_domain from full-label activity_type
# Domain lists from checking_act24.ipynb, extended with new labels (case-insensitive matching)
household = {
    "pc- groom, health-related",
    "pc- other personal care",
    "ha- housework",
    "ha- food prep and cleanup",
    "ha- interior maintenance, repair, & decoration",
    "ha- exterior maintenance, repair, & decoration",
    "ha- lawn, garden and houseplants",
    "ha- animals and pets",
    "ha- household management/other household activities",
    "ha- household management/other household activities",
    "ca- caring for and helping children",
    "ca- caring for and helping adults",
    "eat- eating and drinking, waiting",
}

occupation = {
    "wrk- general",
    "wrk- screen based",
    "wrk- desk/screen based",
    "edu- taking class, research, homework",
    "edu- extracurricular",
}

leisure = {
    "ex- hiking",
    "ex- jogging",
    "ex- other",
    "ex- surfing/water sport",
    "ex- walking",
    "ex- weight training",
    "les- screen based leisure time (tv, video game, computer)",
    "les- socializing, communicating, leisure time not screen",
    "ex- participating in sport, exercise or recreation",
    "ex- attending sport, exercise recreation event, or performance",
    "les- socializing, communicating, non-screen based",
    "les- screen-based (tv, video game, computer, phone)",
}

transportation = {
    "trav- biking",
    "trav- driver (car/truck/motorcycle)",
    "trav- passenger (car/truck/motorcycle)",
    "trav- passenger (bus, train, tram, plane, boat, ship)",
    "trav- walking",
    "trav-walking",
    "trav- general",
}

other = {
    "sl- sleep",
    "org- volunteer",
    "org- volunteer work",
    "org- church, spiritual",
    "pur- purchasing goods and services",
}

label_lower = df["activity_type"].str.lower().str.strip()

conditions = [
    label_lower.isin(household),
    label_lower.isin(occupation),
    label_lower.isin(leisure),
    label_lower.isin(transportation),
    label_lower.isin(other),
    label_lower.isin({"other- non-codable", "other- non codable"}),
]

choices = ["household", "occupation", "leisure", "transportation", "other", "non_codable"]

df["broad_domain"] = np.select(conditions, choices, default="unmapped")

# Where activity_type is NaN, broad_domain should also be NaN
df.loc[df["activity_type"].isna(), "broad_domain"] = np.nan

print("activity_type sample:")
print(df["activity_type"].value_counts(dropna=False).head(15))
print("\nbroad_domain distribution:")
print(df["broad_domain"].value_counts(dropna=False))
unmapped = df.loc[df["broad_domain"] == "unmapped", "activity_type"].value_counts(dropna=False)
print("\nUnmapped activity_type values:")
print(unmapped)


activity_type sample:
activity_type
WRK- Desk/Screen Based                                 113149
EX- Participating in Sport, Exercise or Recreation      82514
WRK- General                                            67168
EAT- Eating and Drinking, Waiting                       28638
LES- Screen-Based (TV, Video Game, Computer, Phone)     26188
TRAV- Driver (Car/Truck/Motorcycle)                     23158
CA- Caring for and Helping Children                     22249
EDU- Taking Class, Research, Homework                   18606
HA- Household Management/Other Household Activities     17764
TRAV- Walking                                           15996
LES- Socializing, Communicating, Non-Screen Based       14562
HA- Food Prep and Cleanup                               14312
OTHER- Non-Codable                                      12418
PUR- Purchasing Goods and Services                       6984
TRAV- Passenger (Car/Truck/Motorcycle)                   6655
Name: count, dtype: int64

broad_d

In [3]:
# Cascade: if activity_type == "OTHER- Non-Codable", all columns broad_domain through intensity_do
# become non_codable (broad.behavior_do uses non_pa as its non_codable equivalent)
_nc = df["activity_type"] == "OTHER- Non-Codable"

df.loc[_nc, "broad_domain"]     = "non_codable"
df.loc[_nc, "broad.behavior_do"] = "non_pa"
df.loc[_nc, "posture_wbm"]      = "non_codable"
df.loc[_nc, "posture_broad"]    = "non_codable"
df.loc[_nc, "broad.posture_do"] = "non_codable"
df.loc[_nc, "sed.posture_do"]   = "non_codable"
df.loc[_nc, "intensity_do"]     = "non_codable"

print(f"Non-codable cascade applied to {_nc.sum():,} rows")
print(df.loc[_nc, ["broad_domain","broad.behavior_do","posture_wbm","posture_broad","broad.posture_do","sed.posture_do","intensity_do"]].iloc[:3])


intensity_do distribution where activity_type == OTHER- Non-Codable:
intensity_do
non_codable    12418
Name: count, dtype: int64


In [4]:
# Export
output_path = r"C:\Users\HELIOS-300\Desktop\WAVES\ACT24 Full Code\act24_testing.csv"
df.to_csv(output_path, index=False)
print(f"Exported {len(df):,} rows to {output_path}")


Exported 490,080 rows to C:\Users\HELIOS-300\Desktop\WAVES\ACT24 Full Code\act24_testing.csv


In [5]:
import pandas as pd
import os
import re

# -------------------------------------------------------------------
# Load the base file (output from earlier pipeline)
# -------------------------------------------------------------------
base_path = r"C:\Users\HELIOS-300\Desktop\WAVES\ACT24 Full Code\act24_testing.csv"
base_df = pd.read_csv(base_path, low_memory=False)
print(f"Base file loaded: {base_df.shape}")

# -------------------------------------------------------------------
# Load and concat all activPal files whose ID exists in the base file
# -------------------------------------------------------------------
activpal_folder = r"C:\Users\HELIOS-300\Desktop\Data\activPal ACT24"
valid_ids = set(base_df["id"].unique())

activpal_chunks = []

for fname in sorted(os.listdir(activpal_folder)):
    m = re.match(r"ACT24_(\d+)", fname, re.IGNORECASE)
    if not m:
        continue
    file_id = int(m.group(1))
    if file_id not in valid_ids:
        continue

    path = os.path.join(activpal_folder, fname)
    ap = pd.read_csv(path, sep=";", skiprows=1, low_memory=False)

    # Drop the raw numeric Time column, keep Time(approx) as the join key
    ap = ap.drop(columns=["Time"], errors="ignore")
    ap = ap.rename(columns={"Time(approx)": "date_time"})

    # Add id so we can join on both id + date_time
    ap["id"] = file_id

    activpal_chunks.append(ap)
    print(f"  Loaded ID={file_id:3d} | {len(ap):>9,} rows")

activpal_df = pd.concat(activpal_chunks, ignore_index=True)
print(f"\nAll activPal combined: {activpal_df.shape}")
print("activPal columns:", activpal_df.columns.tolist())


Base file loaded: (490080, 15)
  Loaded ID=102 |   673,926 rows


  Loaded ID=116 |   198,247 rows
  Loaded ID=117 |   850,154 rows
  Loaded ID=122 |   930,718 rows
  Loaded ID=124 |   682,444 rows
  Loaded ID=126 |   517,394 rows
  Loaded ID=127 | 1,209,602 rows
  Loaded ID=128 |   688,080 rows
  Loaded ID=129 | 1,193,267 rows
  Loaded ID=130 | 1,031,785 rows
  Loaded ID=131 |   775,974 rows
  Loaded ID=132 |   961,245 rows
  Loaded ID=133 | 1,126,091 rows
  Loaded ID=134 |   601,133 rows
  Loaded ID=136 |   688,985 rows
  Loaded ID=138 |   688,678 rows
  Loaded ID=139 |   964,663 rows
  Loaded ID=140 |   697,671 rows
  Loaded ID=141 |   512,273 rows
  Loaded ID=143 |   451,410 rows
  Loaded ID=144 |   777,466 rows
  Loaded ID=150 |   954,318 rows
  Loaded ID=154 |   776,949 rows

All activPal combined: (17952473, 18)
activPal columns: ['date_time', 'StepCount', 'Activity Score (MET.s)', 'Sedentary Time (s)', 'Upright Time (s)', 'Stepping Time (s)', 'Cycling Time (s)', 'Primary Lying Time (s)', 'Secondary Lying Time (s)', 'Nonwear Time (s)', 'Seated

In [6]:
# -------------------------------------------------------------------
# Left merge: keep every row from base_df, bring in activPal columns
# Join key: id + date_time (both already in YYYY-MM-DD HH:MM:SS format)
# -------------------------------------------------------------------
merged_df = base_df.merge(activpal_df, on=["id", "date_time"], how="left")

print(f"Merged shape: {merged_df.shape}")
print(f"Base rows:    {len(base_df):,}  (should be unchanged)")
print()

# Coverage: how many base rows got at least one activPal column filled
new_cols = [c for c in activpal_df.columns if c not in ("id", "date_time")]
filled = merged_df[new_cols[0]].notna().sum()
print(f"Rows with activPal data matched: {filled:,} / {len(merged_df):,} ({filled/len(merged_df)*100:.1f}%)")
print()
print("New activPal columns added:", new_cols)


Merged shape: (490080, 31)
Base rows:    490,080  (should be unchanged)

Rows with activPal data matched: 457,920 / 490,080 (93.4%)

New activPal columns added: ['StepCount', 'Activity Score (MET.s)', 'Sedentary Time (s)', 'Upright Time (s)', 'Stepping Time (s)', 'Cycling Time (s)', 'Primary Lying Time (s)', 'Secondary Lying Time (s)', 'Nonwear Time (s)', 'Seated Transport Time (s)', 'Data Errors (s)', 'Sedentary to Upright Movements', 'Upright to Sedentary Movements', 'Sum(abs(dChannel1))', 'Sum(abs(dChannel2))', 'Sum(abs(dChannel3))']


In [7]:
# -------------------------------------------------------------------
# Nullify all activPal columns where activity_type == "OTHER- Non-Codable"
# -------------------------------------------------------------------
activpal_cols = [c for c in merged_df.columns if c not in base_df.columns]

non_codable_mask = merged_df["activity_type"] == "OTHER- Non-Codable"
merged_df.loc[non_codable_mask, activpal_cols] = np.nan

print(f"Rows where activity_type == OTHER- Non-Codable: {non_codable_mask.sum():,}")
print(f"activPal columns nullified for those rows: {activpal_cols}")


Rows where intensity_do == non_codable: 12,418
activPal columns nullified for those rows: ['StepCount', 'Activity Score (MET.s)', 'Sedentary Time (s)', 'Upright Time (s)', 'Stepping Time (s)', 'Cycling Time (s)', 'Primary Lying Time (s)', 'Secondary Lying Time (s)', 'Nonwear Time (s)', 'Seated Transport Time (s)', 'Data Errors (s)', 'Sedentary to Upright Movements', 'Upright to Sedentary Movements', 'Sum(abs(dChannel1))', 'Sum(abs(dChannel2))', 'Sum(abs(dChannel3))']


In [8]:
# -------------------------------------------------------------------
# Export final merged file
# -------------------------------------------------------------------
output_path = r"C:\Users\HELIOS-300\Desktop\WAVES\ACT24 Full Code\act24_testing.csv"
merged_df.to_csv(output_path, index=False)
print(f"Exported {len(merged_df):,} rows to {output_path}")
print("Final columns:", merged_df.columns.tolist())


Exported 490,080 rows to C:\Users\HELIOS-300\Desktop\WAVES\ACT24 Full Code\act24_testing.csv
Final columns: ['id', 'obs', 'date', 'date_time', 'rel_time', 'activity_type', 'broad_domain', 'broad.behavior_do', 'posture_wbm', 'posture_broad', 'broad.posture_do', 'sed.posture_do', 'intensity_do', 'Quality', 'Step', 'StepCount', 'Activity Score (MET.s)', 'Sedentary Time (s)', 'Upright Time (s)', 'Stepping Time (s)', 'Cycling Time (s)', 'Primary Lying Time (s)', 'Secondary Lying Time (s)', 'Nonwear Time (s)', 'Seated Transport Time (s)', 'Data Errors (s)', 'Sedentary to Upright Movements', 'Upright to Sedentary Movements', 'Sum(abs(dChannel1))', 'Sum(abs(dChannel2))', 'Sum(abs(dChannel3))']


In [9]:
# -------------------------------------------------------------------
# Summary export: sedentary time + steps by id + obs (GT vs activPal)
# -------------------------------------------------------------------
def summarise_group(g):
    step_gt   = g["Step"].sum(min_count=1)

    posture_col = g["broad.posture_do"]
    sed_gt = (
        (posture_col == "sedentary").sum()
        if posture_col.notna().any()
        else pd.NA
    )

    step_ap_raw = pd.to_numeric(g["StepCount"], errors="coerce")
    step_ap = step_ap_raw.sum(min_count=1)

    sed_ap_raw = pd.to_numeric(g["Sedentary Time (s)"], errors="coerce")
    sed_ap = (
        (sed_ap_raw == 1).sum()
        if sed_ap_raw.notna().any()
        else pd.NA
    )

    return pd.Series({
        "gt_total_steps":  step_gt,
        "gt_sedentary_s":  sed_gt,
        "ap_total_steps":  step_ap,
        "ap_sedentary_s":  sed_ap,
    })

summary = merged_df.groupby(["id", "obs"]).apply(
    summarise_group, include_groups=False
).reset_index()

print(summary.to_string())

summary_path = r"C:\Users\HELIOS-300\Desktop\WAVES\ACT24 Full Code\summary_act24_testing.csv"
summary.to_csv(summary_path, index=False)
print(f"\nExported summary ({len(summary)} rows) to {summary_path}")


     id  obs  gt_total_steps  gt_sedentary_s  ap_total_steps ap_sedentary_s
0   102    1          4079.0          7816.0          3900.0         6882.0
1   102    2          3397.0          8323.0          3248.0         4583.0
2   116    1          4505.0          2401.0          2994.0         2477.0
3   116    2           425.0          4258.0          4190.0         4208.0
4   117    1          5362.0          5922.0          5200.0         3771.0
5   117    2          3066.0          8685.0          2986.0         1882.0
6   122    1           324.0          7642.0           306.0         4885.0
7   122    2         10916.0          1036.0         10656.0          252.0
8   124    1          2709.0          6435.0          2160.0         5971.0
9   124    2          1880.0          5419.0          1726.0         5371.0
10  126    1             NaN          5438.0           448.0         2821.0
11  126    2          1452.0          9317.0             NaN           <NA>
12  127    1